In [3]:
import os
import sys
import pandas as pd
import yaml 
from matplotlib import pyplot as plt
from matplotlib import ticker as mticker
from matplotlib import colors as mcolors
from matplotlib import patches as mpatches
import statsmodels.api as sm
import numpy as np
from itertools import product
import subprocess

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11
#plt.rcParams['text.usetex'] = True

from sklearn.decomposition import PCA

with open("../../config.yaml.local", "r") as f:
    LOCAL_CONFIG = yaml.safe_load(f)
with open("../../config.yaml", "r") as f:
    CONFIG = yaml.safe_load(f)
sys.path.append("../python")

import globals
import data_tools as dt
import utils
import emb

LOCAL_PATH = LOCAL_CONFIG["LOCAL_PATH"]
RAW_DATA_PATH = LOCAL_CONFIG["RAW_DATA_PATH"]
DATA_PATH = LOCAL_CONFIG["DATA_PATH"]
R_PATH = LOCAL_CONFIG["R_PATH"]

RUN_R_SCRIPTS = False
REPLACE_ANALYSIS_DATA = False

analysis_data_filepath = os.path.join(
    DATA_PATH, "v4v-analysis-2.parquet"
)

TITLE_PCA_K = 10
TEXT_PCA_K = 10


In [4]:
# Making analysis data for R

if not os.path.exists(analysis_data_filepath) or REPLACE_ANALYSIS_DATA:

    df = dt.get_post_quality_analysis_data()
    df = df.loc[df['title'] != 'deleted by author'].reset_index(drop=True)

    # Dynamically process embeddings instead of loading from
    # post_embeddings.pkl to save on memory
    title_embeddings = []
    text_embeddings = []

    for idx, row in df.iterrows():
        title_emb = np.array(emb.get_embedding_robust(row['title']))
        title_emb = title_emb / np.linalg.norm(title_emb)
        title_embeddings.append(title_emb)
        
        text_emb = np.array(emb.get_embedding_robust(row['text']))
        text_emb = text_emb / np.linalg.norm(text_emb)
        text_embeddings.append(text_emb)

    title_embeddings = np.array(title_embeddings)
    text_embeddings = np.array(text_embeddings)    

    # scree plot for title embeddings
    title_pca = PCA()
    title_pca.fit(title_embeddings)
    explained_variance = title_pca.explained_variance_ratio_[0:50]
    plt.figure(figsize=(6, 4))
    plt.plot(range(1, len(explained_variance) + 1), explained_variance, marker='o')
    #plt.title('Title Embeddings: PCA Scree Plot')
    plt.xlabel('Principal Component')
    plt.ylabel('Explained Variance Ratio')
    plt.grid()
    filename = os.path.join(LOCAL_PATH, 'results', 'fig_title_scree_plot.pdf')
    plt.savefig(filename, bbox_inches='tight')
    plt.show()

    # scree plot for text embeddings
    text_pca = PCA()
    text_pca.fit(text_embeddings)
    explained_variance = text_pca.explained_variance_ratio_[0:50]
    plt.figure(figsize=(6, 4))
    plt.plot(range(1, len(explained_variance) + 1), explained_variance, marker='o')
    #plt.title('Text Embeddings: PCA Scree Plot')
    plt.xlabel('Principal Component')
    plt.ylabel('Explained Variance Ratio')
    plt.grid()
    filename = os.path.join(LOCAL_PATH, 'results', 'fig_text_scree_plot.pdf')
    plt.savefig(filename, bbox_inches='tight')
    plt.show()

    # PCA for embeddings
    title_pca = PCA(n_components=TITLE_PCA_K)
    title_pca.fit(title_embeddings)
    title_pca_embeddings = title_pca.transform(title_embeddings)
    for k in range(TITLE_PCA_K):
        df[f'title_emb_{k}'] = title_pca_embeddings[:, k]

    text_pca = PCA(n_components=TEXT_PCA_K)
    text_pca.fit(text_embeddings)
    text_pca_embeddings = text_pca.transform(text_embeddings)
    for k in range(TEXT_PCA_K):
        df[f'text_emb_{k}'] = text_pca_embeddings[:, k]

    # Output analysis data file
    df.to_parquet(analysis_data_filepath, index=False)
    print(f"Saved analysis data to {analysis_data_filepath}")
else:
    df = pd.read_parquet(analysis_data_filepath)
    print(f"Loaded analysis data from {analysis_data_filepath}")



Loaded analysis data from /Users/ekung/Dropbox/projects/sn-research/processed_data/v4v-analysis-2.parquet


In [5]:
# Running initial R regression to get quality measures

res = subprocess.run([R_PATH, LOCAL_PATH + "/src/R/v4v-analysis-2.R"], check=True, capture_output=True, text=True)
print(res.stdout)

[1] 1280
                                  r1                  r2                  r3
Dependent Var.:           log_sats48          log_sats48          log_sats48
                                                                            
Constant           3.253*** (0.0146)   3.200*** (0.0217)                    
log_numwords      0.3222*** (0.0030)  0.2543*** (0.0085)  0.1599*** (0.0083)
is_link_postTRUE -0.4845*** (0.0141) -0.1969*** (0.0174) -0.4614*** (0.0191)
num_img_or_links  0.0199*** (0.0012)  0.0153*** (0.0012)  0.0104*** (0.0013)
text_emb_0                            0.4629*** (0.0379)   -0.0842* (0.0364)
text_emb_1                            0.5504*** (0.0491)  -0.1624** (0.0547)
text_emb_2                           -0.4269*** (0.0504)  0.5047*** (0.0533)
text_emb_3                           -0.4908*** (0.0468)  0.4549*** (0.0550)
text_emb_4                            -1.650*** (0.0581) -0.6009*** (0.0657)
text_emb_5                            -2.673*** (0.0623)  -1.729***

In [7]:
# Generate regression output table

coefs_filepath = os.path.join(DATA_PATH, "v4v-analysis-2-reg-coefs.parquet")
coefs_df = pd.read_parquet(coefs_filepath)


In [ ]:
# Construct zap surprise histories

wq_filepath = os.path.join(DATA_PATH, "v4v-analysis-2-w-q.parquet")
df = pd.read_parquet(wq_filepath)

df = df.sort_values(by=['userId', 'itemId'], ascending=[True, True]).reset_index(drop=True)

# cumsum of zap surprises, less the current one
cumsum = df.groupby('userId')['surprise'].cumsum() - df['surprise']

# count of prior items
count = df.groupby('userId').cumcount()

# Running average of zap surprise
df['avg_surprise'] = cumsum / count.replace(0, np.nan)   



In [16]:
count

0         0
1         1
2         2
3         3
4         4
         ..
175852    0
175853    0
175854    0
175855    1
175856    0
Length: 175857, dtype: int64